# Swapping the Judge — Jev vs. the LLM in `get_next_step()`

Every step, `TaskHandler.get_next_step()` makes one `structured_output()`
call asking the backbone LLM to classify where things stand: `next_step`
or `final_result`. When it picks `final_result`, the LLM's own `content`
is discarded: the actual `TaskResult.content` comes from the *previous*
step's result, not from this call. So on the step that ends the task, a
full structured-output call is spent generating an answer that is thrown
away; only the `kind` label survives.

This notebook replaces that classification with a judgment call to
TypeSafe's Jev model instead. Jev's `Noul` primitive answers exactly this
shape of question: a calibrated yes/no judgment, returned as a
probability rather than a flat label. A `JevTaskHandler` asks Jev whether
the task is done; the LLM is only called to write the next instruction,
and only when Jev says the task isn't done yet. The step that used to
generate and discard an answer now makes no LLM call at all.

**NOTE**
`LLMAgent.run()` builds its own `TaskHandler` internally and has no
parameter for swapping in a subclass, and its driving loop
(`_process_loop`) is a closure with no extension point. This notebook
copies that loop's body into a manual `while not task_handler.done()`
loop driving a `JevTaskHandler` directly, the same pattern
`run_supervised()`'s callers already use for `SupervisedTaskHandler`.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

This notebook uses TypeSafe SDK. To install it, run the command below.

In [2]:
!uv pip install typesafe-sdk -q

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

To use [Ollama Cloud](https://ollama.com/cloud) instead of a local model,
set `OLLAMA_HOST=https://ollama.com` and `OLLAMA_API_KEY` in your
environment before running the cells below.

In [3]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [4]:
model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

## Requirements

- A TypeSafe API key stored in the environment variable `TYPESAFE_API_KEY`,
  created at [console.typesafe.ai](https://console.typesafe.ai)

In [5]:
import getpass

if not os.environ.get("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("TypeSafe API Key: ")

In [6]:
import json
import logging
import os

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)


def get_next_evolution(species_name: str) -> str:
    """Looks up the next evolution stage for a Pok\u00e9mon species.

    Returns the next stage's name, or "no further evolution" if the
    species is already at the end of its evolution chain.
    """
    species_name = species_name.lower().strip()
    req = urllib.request.Request(
        f"https://pokeapi.co/api/v2/pokemon-species/{species_name}",
        headers={"User-Agent": "llm-agents-from-scratch/1.0"},
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        species = json.loads(resp.read())

    chain_req = urllib.request.Request(
        species["evolution_chain"]["url"],
        headers={"User-Agent": "llm-agents-from-scratch/1.0"},
    )
    with urllib.request.urlopen(chain_req, timeout=10) as resp:
        chain = json.loads(resp.read())["chain"]

    def _find(node):
        if node["species"]["name"] == species_name:
            return node
        for child in node["evolves_to"]:
            found = _find(child)
            if found is not None:
                return found
        return None

    node = _find(chain)
    if node is None or not node["evolves_to"]:
        return "no further evolution"
    return node["evolves_to"][0]["species"]["name"]


llm = OllamaLLM(
    host=host,
    model="glm-5.3:cloud" if use_cloud else "qwen3:14b",
    think=use_cloud,
    json_prompt_mode=use_cloud,
)
agent = LLMAgent(
    llm=llm,
    tools=[SimpleFunctionTool(func=get_next_evolution)],
)

TASK_INSTRUCTION = (
    "Starting from charmander, use get_next_evolution (only, not your "
    "prior knowledge) one step at a time to walk its evolution chain "
    "until there is no further evolution, then report the complete "
    "chain."
)

## Baseline: the LLM decides

`agent.run()` drives the task with the stock `TaskHandler`: every step,
`get_next_step()` asks the LLM to classify the current response as
`next_step` or `final_result` in one `structured_output()` call.

In [7]:
baseline_task = Task(instruction=TASK_INSTRUCTION)
baseline_result = await agent.run(baseline_task)
print(baseline_result.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Starting from charmander, use get_next_evolution (only, not your prior knowledge) one step at a time to walk its evolution chain unti...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Starting from charmander, use get_next_evolution (only, not your prior knowledge) one step at a time to walk its evolution chain u...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: get_next_evolution
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: charmeleon
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "e0e2604a-cac3-487a-b46a-52dae573bd5d",
    "tool_name": "get_next_evolution",
  ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the pending tool call: get_next_evolution with arguments {"species_name": "charmeleon"} to continue walking the evolution chain. S...[TRUNCATED]
INFO (llm_agents_fs.TaskH

## Splitting judgment from generation: `JevTaskHandler`

`JevTaskHandler` overrides `get_next_step()`. Instead of one LLM call
that both classifies and (possibly wastefully) generates, it asks Jev's
`Noul` primitive a single yes/no question (is the task done) over a
`state` built from the same rollout, current response, and instruction
the stock prompt uses. Jev returns a probability, compared against
`done_threshold`:

- **Done:** no LLM call at all. The previous step's content becomes the
  `TaskResult` directly, exactly like the stock handler, just without
  the discarded generation.
- **Not done:** the LLM is called once with a continuation-only prompt
  and a schema that only has `content` -- no `kind` field at all, so
  it can't disagree with Jev by classifying `final_result` and
  returning an empty instruction, the way reusing the stock
  `NextStepDecision` schema could.

In [8]:
from typing import Any

from pydantic import BaseModel, Field
from typesafe_sdk import AsyncTypeSafeClient, Noul, NoulCriteria

from llm_agents_from_scratch.data_structures import (
    RejectedTaskResult,
    TaskResult,
    TaskStep,
    TaskStepResult,
)


class NextStepContent(BaseModel):
    """Structured output used by JevTaskHandler.get_next_step()."""

    content: str = Field(
        description="The next tool call or action to take.",
    )


class JevTaskHandler(LLMAgent.TaskHandler):
    """A TaskHandler whose stop/continue judgment comes from Jev.

    Attributes:
        typesafe_client (AsyncTypeSafeClient): Client used to reach Jev.
        done_threshold (float): Minimum Noul probability to treat the
            task as complete.
    """

    def __init__(
        self,
        *args: Any,
        typesafe_client: AsyncTypeSafeClient,
        done_threshold: float = 0.8,
        **kwargs: Any,
    ) -> None:
        """Initialize a JevTaskHandler.

        Args:
            typesafe_client (AsyncTypeSafeClient): Client used to reach
                Jev.
            done_threshold (float): Minimum Noul probability to treat
                the task as complete. Defaults to 0.8.
            *args: Forwarded to TaskHandler.
            **kwargs: Forwarded to TaskHandler.
        """
        super().__init__(*args, **kwargs)
        self.typesafe_client = typesafe_client
        self.done_threshold = done_threshold

    async def get_next_step(
        self,
        previous_step_result: TaskStepResult | RejectedTaskResult | None,
    ) -> TaskStep | TaskResult:
        """Ask Jev whether the task is done; the LLM only writes steps."""
        if not previous_step_result or isinstance(
            previous_step_result,
            RejectedTaskResult,
        ):
            return await super().get_next_step(previous_step_result)

        state = (
            f"<user-instruction>\n{self.task.instruction}\n"
            "</user-instruction>\n\n"
            f"<thinking-process>\n{self.rollout}\n</thinking-process>\n\n"
            f"<current-response>\n{previous_step_result.content}\n"
            "</current-response>"
        )
        response = await self.typesafe_client.system_one(
            state=state,
            questions={
                "is_done": Noul(
                    instructions=(
                        "Given the user's instruction, the assistant's "
                        "rollout so far, and its most recent response, "
                        "decide whether the task is fully complete."
                    ),
                    criteria=NoulCriteria(
                        true=(
                            "The current response is a final answer; no "
                            "further tool calls or steps are needed."
                        ),
                        false=(
                            "The current response describes a pending "
                            "action, a plan, or an intermediate result "
                            "-- more work is needed."
                        ),
                    ),
                ),
            },
            model="jev-latest",
        )
        confidence = response.answers["is_done"].noul
        self.logger.info(
            f"\U0001f52e Jev confidence task is done: {confidence:.2f}",
        )

        if confidence >= self.done_threshold:
            return TaskResult(
                task_id=self.task.id_,
                content=previous_step_result.content,
            )

        prompt = (
            "You are overseeing an assistant's progress in "
            "accomplishing a user instruction. The task is not yet "
            "complete; more steps are needed. Describe the next tool "
            "call or action the assistant should take.\n\n"
            f"<thinking-process>\n{self.rollout}\n</thinking-process>\n\n"
            f"<current-response>\n{previous_step_result.content}\n"
            "</current-response>\n\n"
            f"<user-instruction>\n{self.task.instruction}\n"
            "</user-instruction>"
        )
        next_content = await self.llm_agent.llm.structured_output(
            prompt=prompt,
            mdl=NextStepContent,
        )
        return TaskStep(
            task_id=self.task.id_,
            instruction=next_content.content,
        )

## Driving `JevTaskHandler` manually

This is `_process_loop()`'s body, unchanged, driving a `JevTaskHandler`
instead of the stock one.

We can run a task directly with `JevTaskHandler`, just by passing it an `LLMAgent`
and a `Task` to perform.

In [9]:
from llm_agents_from_scratch.errors import MaxStepsReachedError


async def run_with_jev(
    agent: LLMAgent,
    task: Task,
    typesafe_client: AsyncTypeSafeClient,
    done_threshold: float = 0.8,
    max_steps: int | None = None,
) -> "LLMAgent.TaskHandler":
    """Drives a JevTaskHandler to completion, mirroring _process_loop()."""
    task_handler = JevTaskHandler(
        llm_agent=agent,
        task=task,
        typesafe_client=typesafe_client,
        done_threshold=done_threshold,
    )
    await task_handler.load_memories()

    step_result = None
    while not task_handler.done():
        try:
            if task_handler.step_counter == max_steps:
                raise MaxStepsReachedError("Max steps reached.")
            next_step = await task_handler.get_next_step(step_result)
            match next_step:
                case TaskStep():
                    step_result = await task_handler.run_step(next_step)
                case TaskResult():
                    await task_handler.record_memory(result=next_step)
                    task_handler.set_result(next_step)
        except Exception as e:
            await task_handler.record_memory(error=e)
            task_handler.set_exception(e)

    return task_handler

In [10]:
jev_task = Task(instruction=TASK_INSTRUCTION)

async with AsyncTypeSafeClient() as typesafe_client:
    jev_task_handler = await run_with_jev(
        agent,
        jev_task,
        typesafe_client,
        max_steps=5,
    )

jev_result = jev_task_handler.result()
print(jev_result.content)

INFO (llm_agents_fs.JevTaskHandler) :      ⚙️ Processing Step: Starting from charmander, use get_next_evolution (only, not your prior knowledge) one step at a time to walk its evolution chain u...[TRUNCATED]
INFO (llm_agents_fs.JevTaskHandler) :      🛠️ Executing Tool Call: get_next_evolution
INFO (llm_agents_fs.JevTaskHandler) :      ✅ Successful Tool Call: charmeleon
INFO (llm_agents_fs.JevTaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "3e907141-0f64-4dd9-8edc-e6e46300648b",
    "tool_name": "get_next_evolution{\"sp...[TRUNCATED]
INFO (llm_agents_fs.JevTaskHandler) :      🔮 Jev confidence task is done: 0.02
INFO (llm_agents_fs.JevTaskHandler) :      ⚙️ Processing Step: The previous tool call was malformed—the tool name and arguments were incorrectly merged into a single string. Discard that call a...[TRUNCATED]
INFO (llm_agents_fs.JevTaskHandler) :      🛠️ Executing Tool Call: get_next_evolution
INFO (llm_agents_fs.JevTaskHandler) :      ✅ Su

## Comparing the two runs

Both runs reach the same answer, driven by the same backbone LLM for
tool use and step generation. What changed is the next-step/stopping
decision:

- The original `TaskHandler` asks the LLM to classify *and* generate
  on every step, discarding the generation whenever the classification
  is `final_result`.
- `JevTaskHandler` asks Jev for a calibrated probability first, and
  only calls the LLM to generate when there's genuinely something left
  to write.
- `done_threshold` turns that probability into a tunable policy: raise
  it to make the agent more conservative about declaring a task
  finished, lower it to stop sooner, a lever the binary `kind` label
  never offered.

It's not hard to think of where else Jev may be used in the processing
loop. Essentially anywhere a structured decision needs to be made is a
candidate application for Jev. Looking ahead to capabilities future
chapters enable:

- Jev could judge whether a skill should be activated before the
  single-turn chat with the backbone LLM even starts, rather than
  leaving that decision to the LLM's own `use_skill` tool call
  (Chapter 6).
- Jev could judge whether a subagent should be dispatched for a given
  step, rather than leaving that decision to the LLM's own
  `use_subagent` tool call (Chapter 9).